# 🔄 02 — Preprocessing & Augmentation

Demonstrates the data preprocessing and augmentation pipeline:
- Image resizing and normalization
- Noise reduction
- Data augmentation techniques
- Before/after visualization

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

import config
from src.data.preprocessing import preprocess_image, split_dataset
from src.data.augmentation import get_augmentation_config, create_train_generator

%matplotlib inline
plt.style.use('dark_background')
print('✓ Libraries loaded')

## 1. Dataset Splitting
If you have raw data in `data/raw/Normal/` and `data/raw/Cancerous/`, split it:

In [ ]:
# Uncomment to split your raw dataset:
# stats = split_dataset(
#     source_dir=str(config.RAW_DATA_DIR),
#     output_base_dir=str(config.PROCESSED_DATA_DIR),
#     train_ratio=0.70,
#     val_ratio=0.15,
#     test_ratio=0.15,
# )
# print(stats)
print('Uncomment the cell above to split your dataset.')

## 2. Preprocessing Demo
Demonstrate resizing, normalization, and noise reduction on a sample image.

In [ ]:
# Create a synthetic demo image if no real data available
demo_img = np.random.randint(50, 200, (400, 500, 3), dtype=np.uint8)
demo_path = str(config.DATA_DIR / 'demo_image.png')
Image.fromarray(demo_img).save(demo_path)

# Preprocess with different settings
original = preprocess_image(demo_path, target_size=(224, 224), normalize=False, denoise=False)
normalized = preprocess_image(demo_path, target_size=(224, 224), normalize=True, denoise=False)
denoised = preprocess_image(demo_path, target_size=(224, 224), normalize=True, denoise=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(original.astype(np.uint8))
axes[0].set_title('Resized (224×224)', fontweight='bold')
axes[1].imshow(normalized)
axes[1].set_title('Normalized [0, 1]', fontweight='bold')
axes[2].imshow(denoised)
axes[2].set_title('Denoised + Normalized', fontweight='bold')
for ax in axes: ax.axis('off')
plt.suptitle('Preprocessing Pipeline', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Original shape: {original.shape}, dtype: {original.dtype}')
print(f'Normalized range: [{normalized.min():.3f}, {normalized.max():.3f}]')

## 3. Data Augmentation Demo
Visualize augmentation transforms applied to a single image.

In [ ]:
aug_config = get_augmentation_config()
print('Augmentation Configuration:')
for key, value in aug_config.items():
    print(f'  {key}: {value}')

In [ ]:
# Apply augmentation to a single image
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
)

img = load_img(demo_path, target_size=(224, 224))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes[0][0].imshow(img_array[0].astype(np.uint8))
axes[0][0].set_title('Original', fontweight='bold', color='#00cec9')

aug_labels = ['Rotated', 'Shifted', 'Zoomed', 'Flipped',
              'Bright+', 'Sheared', 'Combined', 'Combined', 'Combined']

aug_iter = datagen.flow(img_array, batch_size=1)
for idx in range(9):
    row = (idx + 1) // 5
    col = (idx + 1) % 5
    aug_img = next(aug_iter)[0].astype(np.uint8)
    axes[row][col].imshow(aug_img)
    axes[row][col].set_title(aug_labels[idx], fontweight='bold')

for ax_row in axes:
    for ax in ax_row:
        ax.axis('off')

fig.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Pixel Intensity Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
channel_names = ['Red', 'Green', 'Blue']
channel_colors = ['#e17055', '#00b894', '#0984e3']

for i in range(3):
    axes[i].hist(original[:, :, i].flatten(), bins=50, color=channel_colors[i],
                 alpha=0.8, edgecolor='white', linewidth=0.3)
    axes[i].set_title(f'{channel_names[i]} Channel', fontweight='bold')
    axes[i].set_xlabel('Pixel Value')
    axes[i].set_ylabel('Frequency')

fig.suptitle('Pixel Intensity Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
**Next:** Proceed to `03_model_training.ipynb` for model training.